> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 10 · MCP AT RUNTIME</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Voice agents on the Sarvam MCP server</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">14 tools, zero wrappers · tool-choice accuracy · the latency price · when NOT to use MCP</div>
</div>

**Time:** 70 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹9 &nbsp;·&nbsp; **Prereq:** Labs 02, 03, 05, 07, 08

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


## The question this lab answers

In Lab 08 you hand-wired a voice agent: `listen()` called the STT SDK, `think()`
called chat completions, `speak()` called the TTS SDK. Every hop was code you wrote
and own.

The Sarvam MCP server offers a different deal. **Fourteen tools, described to the
model in a standard protocol, and the agent decides which to call.** You write no
wrappers at all.

That trade is not free, and the honest question is not *"does MCP work?"* — it does —
but **"what does it cost me, and where is that price worth paying?"** By the end of
this lab you will have measured three things and be able to answer it yourself:

1. **Tool-choice accuracy** — how often does the agent pick the right tool?
2. **Latency** — what does the MCP round-trip add versus a direct SDK call?
3. **Token cost** — what do 14 tool schemas cost you on every single turn?

> **Setup.** This lab shells out to `uvx sarvam-mcp`. If `uvx` is missing,
> `pip install uv` provides it. Section 0 checks before anything else runs.

---
## 0 · Health check — fail here, not thirty cells later

In [3]:
# ── Is the MCP server actually runnable on this machine? ─────────────────
import shutil, subprocess, os, sys

MCP_OK = False
if shutil.which("uvx") is None:
    print("✕ `uvx` not found on PATH.")
    print("  Fix:  pip install uv          (uvx ships inside the `uv` package)")
else:
    try:
        # ⚠️ stdin must be its own PIPE, not inherited. Without it the child
        # inherits the Jupyter kernel's own stdin — which is typically already
        # closed / EOF in a notebook (no real attached terminal). A stdio MCP
        # server reads EOF on its first stdin read and exits CLEANLY (code 0),
        # which looks identical to "the server is broken" if you only check
        # whether it exited. The real MCP client (MultiServerMCPClient, used
        # below) always gives the child a real pipe — this check must too.
        p = subprocess.Popen(
            ["uvx", "sarvam-mcp"],
            env={**os.environ, "SARVAM_API_KEY": API_KEY},
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True,
        )        
        try:
            # A stdio server WAITS for JSON-RPC on stdin. Exiting early = broken.
            p.wait(timeout=4.0)
            print(f"✕ server exited early (code {p.returncode})")
            print((p.stderr.read() or "")[:400])
        except subprocess.TimeoutExpired:
            print("✓ `uvx sarvam-mcp` starts and waits for input — exactly right.")
            p.terminate(); MCP_OK = True
    except Exception as e:
        print(f"✕ {type(e).__name__}: {e}")

print(f"\nMCP_OK = {MCP_OK}")
if not MCP_OK:
    print("Sections 1-5 need this. Section 6 (the decision framework) reads fine without it.")

✓ `uvx sarvam-mcp` starts and waits for input — exactly right.

MCP_OK = True


In [4]:
# pip install langchain-mcp-adapters langchain-openai langgraph
try:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    from langchain_openai import ChatOpenAI
    # from langgraph.prebuilt import create_react_agent
    HAVE_DEPS = True
except ImportError as e:
    HAVE_DEPS = False
    print("Missing dependency:", e)
    print("Install with:")
    print("  pip install langchain-mcp-adapters langchain-openai langgraph")

SERVER = {
    "sarvam": {
        "command":   "uvx",
        "args":      ["sarvam-mcp"],
        "transport": "stdio",
        "env":       {"SARVAM_API_KEY": API_KEY},
    }
}

# The LLM that will DRIVE the tools. Same base-URL swap as Lab 10.
if HAVE_DEPS:
    llm = ChatOpenAI(
        model      = "sarvam-105b",
        base_url   = "https://api.sarvam.ai/v1",
        api_key    = API_KEY,
        temperature= 0.1,          # low — we want deterministic tool choice
        max_tokens = 4000,
        reasoning_effort = None,    # now a declared field on ChatOpenAI
        # model_kwargs = {"reasoning_effort": None},
    )
    print("LLM ready — sarvam-105b via the OpenAI-compatible endpoint")

LLM ready — sarvam-105b via the OpenAI-compatible endpoint


---
## 1 · What is actually on the other end of that socket?

Never bind tools you have not read. Print every tool the server exposes, with its
schema, before you let a model call any of them.

In [5]:
tools = []
if MCP_OK and HAVE_DEPS:
    # ⚠️ langchain-mcp-adapters 0.3+ always converts a successful MCP result
    # into a LIST of content blocks ([{"type": "text", "text": ...}]), and
    # langchain-openai passes that list straight through as the `tool`
    # message's `content`. Sarvam's chat endpoint requires `content` to be a
    # plain string there and 400s with "Input should be a valid string" —
    # this is the exact bug Lab 09 chased down; same library, same fix here.
    from mcp.types import CallToolResult
    from langchain_core.messages import ToolMessage

    async def stringify_tool_content(request, handler):
        result = await handler(request)
        # Cover BOTH outcomes — a genuinely FAILED tool call still returns
        # list-shaped content if only the success path is handled.
        if isinstance(result, CallToolResult):
            text = "\n".join(
                block.text for block in result.content
                if getattr(block, "type", None) == "text"
            )
            # request.runtime is the injected ToolRuntime — carries the real
            # tool_call_id so the returned ToolMessage still links correctly.
            tool_call_id = getattr(request.runtime, "tool_call_id", None) or ""
            status = "error" if result.isError else "success"
            return ToolMessage(content=text or f"({status}, no text content)",
                               tool_call_id=tool_call_id, status=status)
        return result

    mcp = MultiServerMCPClient(SERVER, tool_interceptors=[stringify_tool_content])
    try:
        tools = await mcp.get_tools()
        print(f"{len(tools)} tools exposed\n")
        print(f"{'tool':<34}{'description':<60}")
        print("─" * 94)
        for t in sorted(tools, key=lambda x: x.name):
            desc = (t.description or "").split("\n")[0][:58]
            print(f"{t.name:<34}{desc:<60}")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")
        print("If this hangs or errors, re-check section 0 and your SARVAM_API_KEY.")
else:
    print("Skipped — MCP unavailable. Read on; the numbers below are the point.")


30 tools exposed

tool                              description                                                 
──────────────────────────────────────────────────────────────────────────────────────────────
sarvam_code_api_reference         Build-time tool — helps write code that uses Sarvam. For r  
sarvam_code_languages             Build-time tool — helps write code that uses Sarvam. For r  
sarvam_code_pricing               Build-time tool — helps write code that uses Sarvam. For r  
sarvam_code_recommend_model       Build-time tool — helps write code that uses Sarvam. For r  
sarvam_code_snippet               Build-time tool — helps write code that uses Sarvam. For r  
sarvam_code_speakers              Build-time tool — helps write code that uses Sarvam. For r  
sarvam_code_validate_request      Build-time tool — helps write code that uses Sarvam. For r  
sarvam_tools_dub                  Runtime tool — calls Sarvam API now. For code-writing help  
sarvam_tools_identify_language  

In [6]:
# The two namespaces, and why the split matters
if tools:
    runtime = [t for t in tools if not t.name.startswith("sarvam_code")]
    builder = [t for t in tools if t.name.startswith("sarvam_code")]
    print(f"RUNTIME tools  ({len(runtime)}) — these DO things, and cost rupees")
    for t in runtime: print("   ", t.name)
    print(f"\nBUILDER tools  ({len(builder)}) — these return docs/snippets, ~free")
    for t in builder: print("   ", t.name)
    print("\nA production voice agent should be given the RUNTIME tools only.")
    print("Handing it the builder tools invites it to read docs mid-call.")

RUNTIME tools  (23) — these DO things, and cost rupees
    sarvam_tools_set_api_key
    sarvam_tools_upgrade
    sarvam_tools_stt_transcribe
    sarvam_tools_stt_translate
    sarvam_tools_stt_batch_submit
    sarvam_tools_stt_batch_status
    sarvam_tools_tts_speak
    sarvam_tools_tts_stream
    sarvam_tools_translate
    sarvam_tools_transliterate
    sarvam_tools_identify_language
    sarvam_tools_text_analytics
    sarvam_tools_llm_complete
    sarvam_tools_vision_extract
    sarvam_tools_vision_job_status
    sarvam_tools_pronunciation_list
    sarvam_tools_pronunciation_get
    sarvam_tools_pronunciation_create
    sarvam_tools_pronunciation_delete
    sarvam_tools_voice
    sarvam_tools_dub
    sarvam_tools_localize
    sarvam_tools_recall

BUILDER tools  (7) — these return docs/snippets, ~free
    sarvam_code_api_reference
    sarvam_code_languages
    sarvam_code_speakers
    sarvam_code_pricing
    sarvam_code_snippet
    sarvam_code_recommend_model
    sarvam_code_validate_

In [7]:
# Read one schema in full — this is what the model actually sees each turn
if tools:
    import json as _json
    stt = next((t for t in tools if "stt" in t.name or "transcribe" in t.name), tools[0])
    print("TOOL:", stt.name)
    print("\nDESCRIPTION:\n", stt.description)
    schema = getattr(stt, "args_schema", None)
    if schema is not None:
        try:
            print("\nARGS SCHEMA:")
            print(_json.dumps(schema.model_json_schema()
                              if hasattr(schema, "model_json_schema") else schema,
                              indent=2)[:1200])
        except Exception as e:
            print("  (schema not renderable:", e, ")")

TOOL: sarvam_tools_stt_transcribe

DESCRIPTION:
 Runtime tool — calls Sarvam API now. For code-writing help, use sarvam_code_* tools.

Transcribe an audio file in any of 23 Indian languages using Saaras v3.

Saaras v3 supports multiple output modes via the `mode` parameter:
  • `transcribe` (default) — standard transcription in the original language
  • `translate` — speech from any Indic language directly to English text
  • `verbatim` — exact word-for-word, no normalization, filler words preserved
  • `translit` — romanization to Latin/Roman script
  • `codemix` — English words in English, Indic words in native script

The default `language_code='unknown'` auto-detects, but specifying the language (e.g. `hi-IN`, `ta-IN`) gives better accuracy.
For very long files (>30s), prefer `sarvam_stt_batch_submit`.

ARGS SCHEMA:
{
  "additionalProperties": false,
  "properties": {
    "audio_path": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "n

> **The hidden bill.** Every one of those schemas is injected into the prompt on
> **every turn**. Fourteen tools is roughly 1,500–2,500 input tokens *before your
> user has said anything*. On a 20-turn call that is real money, and section 5
> measures exactly how much.

---
## 2 · The voice loop, with no plumbing

Lab 08's `VoiceAgent` was ~60 lines of `listen`/`think`/`speak`. Here the agent is
three lines, and it chooses its own tools.

In [8]:
# Make a caller utterance to feed the agent (same trick as Lab 08)
from sarvamai.play import save
from pathlib import Path

def ensure_audio(fname, text, speaker="aditya", lang="hi-IN"):
    p = DATA / fname
    if p.exists() and p.stat().st_size > 44:
        return p
    a = client.text_to_speech.convert(text=text, language_code=lang,
                                      model="bulbul:v3", speaker=speaker)
    save(a, str(p)); cost.tts(len(text), v3=False)
    print(f"  generated {fname}")
    return p

CALLER = "नमस्ते, मेरा EMI कब देय है?"
ensure_audio("mcp_caller.wav", CALLER)
print("caller audio ready:", (DATA / 'mcp_caller.wav').resolve())

  generated mcp_caller.wav
caller audio ready: /workspace/Sarvam_AI/Labs/data/mcp_caller.wav


In [9]:
# ── The entire agent. Three lines. ───────────────────────────────────────

from langchain.agents import create_agent

if tools and HAVE_DEPS:
    agent = create_agent(llm, tools=tools)

    prompt = (f"Transcribe the Hindi audio file at {(DATA/'mcp_caller.wav').resolve()} "
              f"using the speech-to-text tool, then tell me in English what the caller asked.")
    try:
        out = await agent.ainvoke({"messages": [{"role": "user", "content": prompt}]})
        for m in out["messages"]:
            role = getattr(m, "type", "?")
            txt  = (getattr(m, "content", "") or "")
            calls = getattr(m, "tool_calls", None)
            if calls:
                for c in calls:
                    print(f"  [tool_call] {c['name']}({str(c.get('args'))[:80]})")
            elif txt:
                print(f"  [{role}] {str(txt)[:220]}")
        # MCP tool results do NOT report usage back — see section 3 for the fix
        cost.llm(2200, 300)
    except Exception as e:
        print(f"{type(e).__name__}: {e}")
else:
    print("Skipped — MCP unavailable.")

  [human] Transcribe the Hindi audio file at /workspace/Sarvam_AI/Labs/data/mcp_caller.wav using the speech-to-text tool, then tell me in English what the caller asked.
  [tool_call] sarvam_tools_stt_transcribe({'audio_path': '/workspace/Sarvam_AI/Labs/data/mcp_calle)
  [tool] {"transcript":"नमस्ते, मेरा ईएमआई कब बेय है?","language_code":"hi-IN","language_probability":null,"diarized_transcript":null,"timestamps":null,"observability":{"latency_ms":659.1,"upstream_calls":1,"request_ids":["202608
  [ai] The Hindi audio was transcribed as:

**"नमस्ते, मेरा ईएमआई कब बेय है?"**

In English, the caller asked:

> **"Hello, when is my EMI due?"**

It appears to be a customer calling about when their monthly equated installmen


**What just happened.** You did not write `listen()`. You did not import the STT
client. You described a goal in English and the agent found `sarvam_stt_transcribe`
in the tool list, called it with the right path and language, read the transcript
and answered. That is the entire value proposition of MCP in one cell.

Now we find out what it cost.

---
## 3 · The gotcha — MCP tool calls are invisible to your cost meter

Your `CostMeter` is fed from `response.usage`. **MCP tool results do not carry usage
back through the adapter.** So an agent that transcribes ten minutes of audio and
speaks ten replies reports ₹0.00 and your meter quietly lies to you.

Cause it, see it, then fix it.

In [10]:
# ── The failure: a fresh meter, a real tool call, and a zero bill ────────
class NaiveMeter:
    def __init__(self): self.total = 0.0
    def add(self, r): self.total += r

naive = NaiveMeter()
print("A tool call runs, audio is transcribed, rupees are genuinely spent...")
print(f"naive meter says: ₹{naive.total:.4f}   ← WRONG")
print("\nThe SDK call inside the MCP server billed you. Nothing told your notebook.")

A tool call runs, audio is transcribed, rupees are genuinely spent...
naive meter says: ₹0.0000   ← WRONG

The SDK call inside the MCP server billed you. Nothing told your notebook.


In [11]:
# ── The fix: wrap the tool, meter it yourself ────────────────────────────
import time, wave

def duration(path):
    with wave.open(str(path), "rb") as w:
        return w.getnframes() / w.getframerate()

def metered(tool, meter):
    """Wrap an MCP tool so every invocation lands on the cost meter.

    We cannot see the server's token counts, so we bill from the INPUT we sent —
    audio seconds for STT, characters for TTS/translate. Conservative by design.
    """
    original = tool.coroutine or tool.func

    async def _wrapped(**kwargs):
        t0 = time.perf_counter()
        result = await original(**kwargs) if tool.coroutine else original(**kwargs)
        dt = time.perf_counter() - t0

        name = tool.name
        try:
            if "stt" in name or "transcribe" in name:
                fp = kwargs.get("file_path") or kwargs.get("audio_path") or kwargs.get("file")
                if fp and Path(str(fp)).exists():
                    meter.stt(duration(Path(str(fp))))
            elif "tts" in name or "speak" in name:
                meter.tts(len(str(kwargs.get("text", ""))), v3=False)
            elif "translate" in name or "transliterate" in name:
                meter.text(len(str(kwargs.get("input", kwargs.get("text", "")))),
                           kind="translate")
        except Exception:
            pass                              # never let metering break the agent
        print(f"     [metered] {name} · {dt*1000:.0f} ms")
        return result

    tool.coroutine = _wrapped
    return tool

if tools:
    tools = [metered(t, cost) for t in tools]
    print(f"wrapped {len(tools)} tools — every call now hits the ₹ meter")

wrapped 30 tools — every call now hits the ₹ meter


> **The general lesson, beyond Sarvam.** The moment you put a protocol boundary
> between your code and a billed API, your observability stops at that boundary.
> Whatever you cannot see, you cannot cost — and whatever you cannot cost will
> surprise you at scale. Meter at the boundary you *do* control.

---
## 4 · Does it pick the right tool? Measure, do not assume.

This is Lab 07's eval harness, pointed at tool selection. Ten utterances, each with
the tool that *should* fire.

In [12]:
CASES = [
    {"say": "Transcribe the Hindi audio at ./data/mcp_caller.wav",
     "expect": "stt"},
    {"say": "Say 'आपका स्वागत है' out loud in Hindi and save it",
     "expect": "tts"},
    {"say": "Translate 'What is my loan balance?' into Tamil",
     "expect": "translate"},
    {"say": "Convert the name 'भावेश' into Roman script",
     "expect": "translit"},
    {"say": "What language is this text: 'என் கடன் தவணை எப்போது?'",
     "expect": "identify"},
    {"say": "Summarise in one line: an NBFC customer asking about EMI dates",
     "expect": None},
]

def bucket(tool_name):
    """Map a concrete tool name onto the coarse capability we expected."""
    n = tool_name.lower()
    if "transcribe" in n or "stt" in n:      return "stt"
    if "tts" in n or "speak" in n:            return "tts"
    if "transliterate" in n:                  return "translit"
    if "translate" in n:                      return "translate"
    if "identify" in n or "language" in n:    return "identify"
    if "llm" in n or "complete" in n:         return "None"
    return n

In [13]:
# Run the eval. Each case is a fresh agent invocation with seed-equivalent settings.
results = []
if tools and HAVE_DEPS:
    ev_agent = create_agent(llm, tools=tools)
    for c in CASES:
        picked = None
        try:
            out = await ev_agent.ainvoke({"messages": [{"role": "user", "content": c["say"]}]})
            for m in out["messages"]:
                for call in (getattr(m, "tool_calls", None) or []):
                    picked = picked or bucket(call["name"])
            cost.llm(2200, 200)
        except Exception as e:
            picked = f"ERROR:{type(e).__name__}"
        ok = (picked == c["expect"])
        results.append({"say": c["say"][:44], "expected": c["expect"],
                        "picked": picked, "ok": ok})
        print(f"  {'✓' if ok else '✕'}  expected={str(c['expect']):<10} got={str(picked):<12}")

    hits = sum(r["ok"] for r in results)
    print(f"\nTOOL-CHOICE ACCURACY: {hits}/{len(results)} = {hits/len(results):.0%}")
else:
    print("Skipped — MCP unavailable.")

     [metered] sarvam_tools_stt_transcribe · 1973 ms
  ✓  expected=stt        got=stt         
     [metered] sarvam_tools_tts_speak · 1815 ms
     [metered] sarvam_tools_tts_speak · 1043 ms
     [metered] sarvam_tools_tts_speak · 1129 ms
     [metered] sarvam_tools_tts_stream · 2228 ms
  ✓  expected=tts        got=tts         
     [metered] sarvam_tools_translate · 2075 ms
  ✓  expected=translate  got=translate   
     [metered] sarvam_tools_transliterate · 2800 ms
  ✓  expected=translit   got=translit    
     [metered] sarvam_tools_identify_language · 1860 ms
  ✓  expected=identify   got=identify    
  ✓  expected=None       got=None        

TOOL-CHOICE ACCURACY: 6/6 = 100%


**How to read your number.** Anything under ~90% means an agent that occasionally
does the wrong thing to a real customer. The usual causes, in order of frequency:

1. **Too many tools.** Fourteen options with overlapping descriptions is a hard
   choice. Section 5 fixes this.
2. **Vague tool descriptions.** The model only sees the description string.
3. **Ambiguous user phrasing** — which is what real callers produce.

Notice that all three are *context* problems, not model problems. That is the bridge
into Lab 11.

---
## 5 · The two prices: latency and tokens

MCP is a protocol hop. Protocol hops cost milliseconds. For a voice agent working to
an 800 ms budget (Lab 08), milliseconds are the whole game.

In [14]:
# ── Latency: MCP round-trip vs the direct SDK call, same work ────────────
import statistics

def time_direct_stt(path, n=3):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        with open(path, "rb") as f:
            client.speech_to_text.transcribe(file=f, model="saaras:v3",
                                             language_code="hi-IN", mode="transcribe")
        ts.append((time.perf_counter() - t0) * 1000)
        cost.stt(duration(path))
    return ts

async def time_mcp_stt(path, n=3):
    ts = []
    stt_tool = next((t for t in tools if "transcribe" in t.name or "stt" in t.name), None)
    if stt_tool is None: return []
    for _ in range(n):
        t0 = time.perf_counter()
        try:
            fn = stt_tool.coroutine or stt_tool.func
            await fn(file_path=str(Path(path).resolve()), language_code="hi-IN")
        except Exception as e:
            print("   mcp call failed:", type(e).__name__, e); return ts
        ts.append((time.perf_counter() - t0) * 1000)
    return ts

wav = DATA / "mcp_caller.wav"
try:
    direct = time_direct_stt(wav)
    print(f"direct SDK : {statistics.mean(direct):>7.0f} ms   {[f'{t:.0f}' for t in direct]}")
except Exception as e:
    direct = []; print("direct failed:", e)

if tools:
    viamcp = await time_mcp_stt(wav)
    if viamcp:
        print(f"via MCP    : {statistics.mean(viamcp):>7.0f} ms   {[f'{t:.0f}' for t in viamcp]}")
        if direct:
            delta = statistics.mean(viamcp) - statistics.mean(direct)
            print(f"\nMCP overhead: {delta:+.0f} ms per call")
            print(f"As a share of the 800 ms conversational budget: {abs(delta)/800:.0%}")

direct SDK :     927 ms   ['745', '1258', '776']
     [metered] sarvam_tools_stt_transcribe · 1030 ms
     [metered] sarvam_tools_stt_transcribe · 842 ms
     [metered] sarvam_tools_stt_transcribe · 943 ms
via MCP    :     939 ms   ['1031', '842', '944']

MCP overhead: +12 ms per call
As a share of the 800 ms conversational budget: 2%


In [15]:
# ── Tokens: what do 14 tool schemas cost on EVERY turn? ──────────────────
if tools:
    def approx_tokens(s):     # ~4 chars/token for English+JSON
        return len(s) // 4

    per_tool = []
    for t in tools:
        blob = f"{t.name}\n{t.description or ''}"
        sch = getattr(t, "args_schema", None)
        if sch is not None:
            try:
                import json as _j
                blob += _j.dumps(sch.model_json_schema()
                                 if hasattr(sch, "model_json_schema") else sch)
            except Exception:
                pass
        per_tool.append((t.name, approx_tokens(blob)))

    per_tool.sort(key=lambda x: -x[1])
    total = sum(n for _, n in per_tool)
    print(f"{'tool':<34}{'~tokens':>9}")
    print("─" * 43)
    for name, n in per_tool[:8]:
        print(f"{name:<34}{n:>9,}")
    print("─" * 43)
    print(f"{'ALL TOOLS, EVERY TURN':<34}{total:>9,}")

    RATE_IN = 29.28 / 1_000_000
    print(f"\nCost of the tool schemas alone:")
    for turns in (1, 10, 20):
        print(f"  {turns:>2} turn(s): ₹{total*turns*RATE_IN:>7.4f}")
    print(f"  100k calls @ 20 turns: ₹{total*20*RATE_IN*100_000:>12,.0f}")

tool                                ~tokens
───────────────────────────────────────────
sarvam_tools_stt_transcribe             661
sarvam_tools_voice                      627
sarvam_tools_translate                  559
sarvam_tools_tts_speak                  550
sarvam_tools_dub                        537
sarvam_tools_stt_batch_submit           519
sarvam_tools_localize                   479
sarvam_tools_transliterate              476
───────────────────────────────────────────
ALL TOOLS, EVERY TURN                 8,518

Cost of the tool schemas alone:
   1 turn(s): ₹ 0.2494
  10 turn(s): ₹ 2.4941
  20 turn(s): ₹ 4.9881
  100k calls @ 20 turns: ₹     498,814


In [16]:
# ── The fix: give the agent only the tools the job needs ─────────────────
if tools:
    # ⚠️ Filter from `runtime` (section 1's builder-excluded list), not raw
    # `tools` — a substring match against the FULL tool name can pull in a
    # builder/doc tool by accident: "speak" in "sarvam_code_speakers" is True,
    # even though that tool only lists voice names for writing code and never
    # calls anything. That directly violates the rule two cells back ("hand a
    # production agent RUNTIME tools only"). KEEP also now covers every
    # capability the CASES/bucket() above actually test (added transliterate,
    # identify) — the old tuple silently dropped two of six, which would have
    # made "re-run section 4 with `lean`" fail those cases for having no
    # matching tool at all, not for a genuine tool-choice mistake.
    
    # KEEP = ("stt", "transcribe", "tts", "speak", "translate", "transliterate", "identify")
    # lean = [t for t in runtime if any(k in t.name.lower() for k in KEEP)]

    LEAN_NAMES = {
        "sarvam_tools_stt_transcribe",
        "sarvam_tools_tts_speak",
        "sarvam_tools_translate",
        "sarvam_tools_transliterate",
        "sarvam_tools_identify_language",
    }
    lean = [t for t in runtime if t.name in LEAN_NAMES]

    lean_tokens = 0
    for t in lean:
        blob = f"{t.name}\n{t.description or ''}"
        lean_tokens += len(blob) // 4

    print(f"full set : {len(tools):>2} tools")
    print(f"lean set : {len(lean):>2} tools  →  {[t.name for t in lean]}")
    saved = total - lean_tokens
    print(f"\nschema tokens saved per turn : ~{saved:,}")
    print(f"over 100k calls @ 20 turns    : ₹{saved*20*RATE_IN*100_000:,.0f}")
    print("\nFewer tools is not only cheaper — it usually raises tool-choice")
    print("accuracy too, because the model has a smaller decision to make.")
    

full set : 30 tools
lean set :  5 tools  →  ['sarvam_tools_stt_transcribe', 'sarvam_tools_tts_speak', 'sarvam_tools_translate', 'sarvam_tools_transliterate', 'sarvam_tools_identify_language']

schema tokens saved per turn : ~7,900
over 100k calls @ 20 turns    : ₹462,624

Fewer tools is not only cheaper — it usually raises tool-choice
accuracy too, because the model has a smaller decision to make.


> **Try this now:** re-run section 4's eval with `lean` instead of `tools`. Most
> rooms see accuracy go **up** while cost goes **down**. That result surprises
> people, and it is the most useful thing in this lab.

---
## 6 · So when should a voice agent use MCP?

You now have real numbers instead of opinions. Here is how to read them.

In [17]:
cost.report()

TTS          ₹   0.0405  27 chars
LLM          ₹   0.0864  2200 in / 300 out
STT          ₹   0.0256  3.1s
LLM          ₹   0.0791  2200 in / 200 out
TTS          ₹   0.0210  14 chars
TTS          ₹   0.0210  14 chars
TTS          ₹   0.0210  14 chars
TTS          ₹   0.0210  14 chars
LLM          ₹   0.0791  2200 in / 200 out
translate    ₹   0.0480  24 chars
LLM          ₹   0.0791  2200 in / 200 out
translate    ₹   0.0100  5 chars
LLM          ₹   0.0791  2200 in / 200 out
LLM          ₹   0.0791  2200 in / 200 out
LLM          ₹   0.0791  2200 in / 200 out
STT          ₹   0.0256  3.1s
STT          ₹   0.0256  3.1s
STT          ₹   0.0256  3.1s
STT          ₹   0.0256  3.1s
STT          ₹   0.0256  3.1s
STT          ₹   0.0256  3.1s
TOTAL        ₹   0.9224
              (₹1000 free credit → ₹999.08 left)
              Estimated from published rates; actual billing usually lower


0.9224130582010582

| | **Reach for MCP** | **Stay on the direct SDK** |
|---|---|---|
| Latency | Not on the critical path — post-call analysis, batch enrichment | **Live conversation.** Every ms comes out of the 800 ms budget |
| Tools | The set changes often; you want config-driven capability | Fixed, known, three calls you will never change |
| Team | Multiple agents/products share one tool surface | One service, one vendor, one hot path |
| Ops | You want capability without shipping code | You need to see and tune every hop |
| Cost | Volume is modest; developer time dominates | High volume — schema tokens on every turn add up |
| Portability | Swap the MCP server, keep the agent | Portability handled at the base-URL layer (Lab 09) |

**The pattern most production teams land on — and it is not a compromise:**

- **Hot path → direct SDK.** The STT → LLM → TTS loop in Lab 08. Every millisecond
  is visible to the caller, so you own every hop.
- **Everything around it → MCP.** Post-call summarisation, translation of the
  transcript, document lookups, pronunciation-dictionary updates, analytics. None of
  it is latency-critical, and all of it benefits from being configuration rather than
  code.

That is the same build-vs-platform line from Deck 05, drawn one level lower — and now
you can draw it with your own measured numbers instead of a vendor's slide.

---
## ✅ Checkpoint

- [ ] `uvx sarvam-mcp` starts and the tool list printed
- [ ] You can name the two namespaces and say which belongs in a production agent
- [ ] An agent transcribed audio without you writing a single wrapper
- [ ] You caused the silent-cost failure and wrapped the tools to fix it
- [ ] You have a **tool-choice accuracy number** for your own run
- [ ] You have an **MCP latency overhead number** in milliseconds
- [ ] You can state the per-turn token cost of carrying 14 tool schemas

## 🧪 Try this

1. Re-run the section 4 eval with the `lean` tool set. Accuracy up or down? Cost?
2. Rewrite one tool's description to be sharper. Does the tool it competes with stop
   winning? This is prompt engineering applied to *tools*, not messages.
3. Add a second MCP server (a filesystem or HTTP one) alongside Sarvam. Does
   tool-choice accuracy degrade as the pool grows?
4. Put the MCP call **behind** the audio, not in front of it — start playing a filler
   phrase, then call the tool. How much of the overhead disappears perceptually?
5. Take the metering wrapper from section 3 and generalise it into a decorator you
   could publish. That is a genuinely useful open-source contribution.